In [27]:
# ГЕНЕРАЦИЯ

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

main_seed = 13 # Мое любимое число, Господа, не ищите в этом смысла!
np.random.seed(main_seed)

#====================================ИНВЕСТОРЫ==========================================
n_users = 1000      # 1000 инвесторов, ясен пень при недообучении масштабировать можем
n_days = 30         # Наблюдаем за ними 30 дней, здесь аккуратнее, пупсик
start_date = datetime(2026, 6, 1)

user_ids = np.arange(100001, 100001 + n_users)
ages = np.random.randint(18, 65, size=n_users) 
experience_days = np.random.randint(10, 1200, size=n_users) # Опыт инвестора
risk_profiles = np.random.choice(['conservative', 'moderate', 'aggressive'], size=n_users, p=[0.3, 0.5, 0.2])
# Для себя: если кто-то возьмет и захочет реализовать мой проект, лучше добавьте еще парочку профилей


# Статистические данные пользователей 
df_users = pd.DataFrame({
    'user_id': user_ids,
    'age': ages,
    'experience_days': experience_days,
    'risk_profile': risk_profiles
})

# Распределение капитала инвесторов (в рублях, ибо нечего пиндоские бумажки использовать!)
# В среднем в районе 200000 рублей
df_users['portfolio_value_rub'] = np.random.lognormal(mean=12.2, sigma=0.8, size=n_users).round(2)

# Доля акций в портфеле (остальное - защитные облигации и кэш)
# У агрессивных инвесторов акций много, у консервативных - мало.
# Нужно еще понимать, что пока у меня нет реальных данных для обучения, поэтому еще нужна корреляция с возрастом
# Не зря же есть шуточное правило 100 - возраст. Кэтбуст сам заметит когда будут настоящие.

risk = df_users['risk_profile'].values

stocks_ratio = np.where(risk == 'conservative', np.random.uniform(0.05, 0.25, size=n_users),
               np.where(risk == 'moderate',     np.random.uniform(0.30, 0.60, size=n_users),
                                                np.random.uniform(0.65, 0.90, size=n_users)))

stocks_ratio = stocks_ratio - (df_users['age'].values - 18) * 0.003
df_users['stocks_ratio'] = np.clip(stocks_ratio, 0.05, 0.95).round(4)

#=======================================РЫНОЧЕК============================================

dates = [start_date + timedelta(days=i) for i in range(n_days)]
# Обычные колебания индекса в день
market_returns = np.random.normal(0.0005, 0.007, size=n_days) 

# Искусственно создаем сильный обвал рынка акций на 18ое число 
market_returns[17] = -0.075 
# И его паническое продолжение на 19ое число 
# В случае увелечения временного диапазона по-хорошему лучше отредактировать, а то в норму войдет отсутствие штормов 
market_returns[18] = -0.032

df_market = pd.DataFrame({
    'date': dates,
    'imoex_return': market_returns  # Дневная доходность
})

#======================================ЛОГИ=========================================
logs_data = []

for day_idx, date in enumerate(dates):
    m_return = market_returns[day_idx]
    
    for _, user in df_users.iterrows():
        uid = user['user_id']
        age = user['age']
        exp = user['experience_days']
        portfolio = user['portfolio_value_rub']
        stocks_ratio = user['stocks_ratio']
        
        # Базовое число заходов в приложение (Через Пуассон) 
        lambda_opens = 2.0
        
        # Если индекс падает, у людей растет тревожность
        if m_return < -0.01:
            # Молодые и неопытные заходят проверять "красный" портфель гораздо чаще
            anxiety = (65 - age) / 25 * (1000 / (exp + 15)) * stocks_ratio
            lambda_opens += 5.0 * anxiety
            
        app_opens = np.random.poisson(lam=max(1.0, lambda_opens))
        screen_views = app_opens * np.random.randint(2, 5) # просмотры графиков акций
         
        # Вероятность панической продажи акций (фиксации убытка в панике)
        # Акцент на том, что чем чаще человек заходит, тем он более тревожен
        panic_score = 0.0
        if m_return < 0:
            panic_score = (abs(m_return) * stocks_ratio * (app_opens / 2) * (50 / age))
            
        # Сигмоида для расчета вероятности продажи
        panic_prob = 1 / (1 + np.exp(-(-4.5 + panic_score)))
        
        # Флаг: нажал ли кнопку "Продать" для закрытия позиций в минус
        action_panic_sell = np.random.choice([0, 1], p=[1 - panic_prob, panic_prob])
        
        # Оценка бумажного убытка за день 
        day_loss_rub = (portfolio * stocks_ratio * m_return) if m_return < 0 else 0
        
        logs_data.append({
            'date': date,
            'user_id': uid,
            'app_opens': app_opens,
            'screen_views': screen_views,
            'day_loss_rub': round(abs(day_loss_rub), 2),
            'action_panic_sell': action_panic_sell
        })

df_logs = pd.DataFrame(logs_data)

print("Все работает, товарищ!")
print(f"Всего панических продаж зафиксировано: {df_logs['action_panic_sell'].sum()}")

Все работает, товарищ!
Всего панических продаж зафиксировано: 351


In [28]:
#========================Модель===========================
from catboost import CatBoostClassifier, Pool

# Признаки считаем за дни 1-17, таргет берем строго в 18й день
df_pre_crisis_1 = df_logs[df_logs['date'] < dates[17]]
features_1 = df_pre_crisis_1.groupby('user_id').agg(
    mean_daily_opens=('app_opens', 'mean'),
    max_daily_opens=('app_opens', 'max'),
    total_screen_views=('screen_views', 'sum'),
    total_loss_before_crisis=('day_loss_rub', 'sum')
).reset_index()

train_dataset = pd.merge(df_users, features_1, on='user_id')
target_day_1 = df_logs[df_logs['date'] == dates[17]][['user_id', 'action_panic_sell']]
train_dataset = pd.merge(train_dataset, target_day_1, on='user_id')

# ---------------------------------------------------------------------

# Признаки считаем за дни 1-18 (включая первый день паники), таргет берем в 19-й день
df_pre_crisis_2 = df_logs[df_logs['date'] < dates[18]]
features_2 = df_pre_crisis_2.groupby('user_id').agg(
    mean_daily_opens=('app_opens', 'mean'),
    max_daily_opens=('app_opens', 'max'),
    total_screen_views=('screen_views', 'sum'),
    total_loss_before_crisis=('day_loss_rub', 'sum') 
).reset_index()

val_dataset = pd.merge(df_users, features_2, on='user_id')
target_day_2 = df_logs[df_logs['date'] == dates[18]][['user_id', 'action_panic_sell']]
val_dataset = pd.merge(val_dataset, target_day_2, on='user_id')

# =====================================================================
X_train = train_dataset.drop(columns=['user_id', 'action_panic_sell'])
y_train = train_dataset['action_panic_sell']

X_val = val_dataset.drop(columns=['user_id', 'action_panic_sell'])
y_val = val_dataset['action_panic_sell']

cat_features = ['risk_profile'] # единственные категориальные данные

# =====================================================================
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=5,
    eval_metric='AUC',
    random_seed=main_seed,
    verbose=50
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features,
    use_best_model=True
)

model.save_model('Agent Paniker.cbm')

0:	test: 0.6618880	best: 0.6618880 (0)	total: 3.23ms	remaining: 966ms
50:	test: 0.7721328	best: 0.7721328 (50)	total: 132ms	remaining: 644ms
100:	test: 0.8083501	best: 0.8180751 (83)	total: 247ms	remaining: 487ms
150:	test: 0.8321596	best: 0.8371898 (138)	total: 367ms	remaining: 362ms
200:	test: 0.8371898	best: 0.8402079 (175)	total: 490ms	remaining: 241ms
250:	test: 0.8465795	best: 0.8487592 (246)	total: 615ms	remaining: 120ms
299:	test: 0.8452381	best: 0.8542924 (280)	total: 742ms	remaining: 0us

bestTest = 0.8542924212
bestIteration = 280

Shrink model to first 281 iterations.
